# ToolCall M30 standard, full 470M data run

Fixed-purpose Google Colab notebook for `m30_standard_470m_seed42`.

- Starts from random weights when no resume bundle is uploaded.
- Uses the complete `scaling_470m` source-data bundle.
- Pauses cleanly before the Colab allocation limit.
- Exports one resume ZIP containing the latest weights, optimizer,
  scaler, data cursor, RNG state, metrics, and TensorBoard events.
- Does not mount or depend on Google Drive.

Architecture: 10 layers, d=384, 6-head MHA, GELU, 29,990,784 parameters

In [ ]:
# Confirm that Colab assigned a GPU.
import subprocess
subprocess.run(["nvidia-smi"], check=True)

## 1. Upload and install the complete project ZIP

Upload `ToolCall-Extended-Runs-Colab-Complete.zip`. The archive
contains sibling `scaling_runs/` and `extended_runs/` directories.
Re-running this cell in the same runtime reuses the extracted project.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import os, shutil, subprocess, sys, zipfile

WORKSPACE = Path("/content/toolcall_extended_workspace")

def safe_extract_zip(archive: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Unsafe ZIP member: {info.filename}")
        zf.extractall(destination)

existing = WORKSPACE / "scaling" / "extended_runs" / "README.md"
if not existing.is_file():
    print("Upload ToolCall-Extended-Runs-Colab-Complete.zip")
    uploaded = files.upload()
    archives = [
        Path(name) for name in uploaded
        if name.lower().endswith(".zip") and zipfile.is_zipfile(name)
    ]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one valid complete-project ZIP")
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    safe_extract_zip(archives[0], WORKSPACE)

candidates = [
    readme.parent.parent
    for readme in WORKSPACE.rglob("extended_runs/README.md")
    if (readme.parent.parent / "scaling_runs/scaling/__init__.py").is_file()
]
if not candidates:
    raise RuntimeError(
        "Could not find sibling scaling_runs/ and extended_runs/ folders"
    )
SCALING_PARENT = sorted(candidates, key=lambda p: len(p.parts))[0].resolve()
EXTENDED_ROOT = SCALING_PARENT / "extended_runs"
os.chdir(SCALING_PARENT)
print("Scaling parent:", SCALING_PARENT)
print("Extended runs:", EXTENDED_ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(EXTENDED_ROOT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "extended_runs.validate"],
    cwd=SCALING_PARENT,
    check=True,
)

## 2. Upload the 470M data archive

Upload your already-generated `scaling_470m.zip`, `.tar.gz`, or
`.tgz`. The notebook finds the directory containing `COMPLETE`,
moves it to `extended_runs/data/scaling_470m`, and runs the full
bundle verifier. The tokenizer is already inside this data bundle.

In [ ]:
import tarfile, tempfile

DATA_ROOT = EXTENDED_ROOT / "data" / "scaling_470m"

def safe_extract_tar(archive: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive) as tf:
        for member in tf.getmembers():
            target = (destination / member.name).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe TAR member: {member.name}")
        tf.extractall(destination)

if not (DATA_ROOT / "COMPLETE").is_file():
    print("Upload scaling_470m.zip, scaling_470m.tar.gz, or scaling_470m.tgz")
    uploaded = files.upload()
    names = [Path(name) for name in uploaded]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one data archive")
    archive = names[0]
    unpack = Path(tempfile.mkdtemp(prefix="scaling_data_", dir="/content"))
    if zipfile.is_zipfile(archive):
        safe_extract_zip(archive, unpack)
    elif tarfile.is_tarfile(archive):
        safe_extract_tar(archive, unpack)
    else:
        raise RuntimeError(f"Unsupported or invalid archive: {archive}")

    candidates = [
        marker.parent for marker in unpack.rglob("COMPLETE")
        if (marker.parent / "train/shards").is_dir()
        and (marker.parent / "validation_general/shards").is_dir()
        and (marker.parent / "validation_structured/shards").is_dir()
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one complete scaling bundle, found {len(candidates)}"
        )
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if DATA_ROOT.exists():
        raise RuntimeError(
            f"Refusing to replace incomplete existing directory: {DATA_ROOT}"
        )
    shutil.move(str(candidates[0]), str(DATA_ROOT))
    shutil.rmtree(unpack, ignore_errors=True)

subprocess.run(
    [
        sys.executable,
        str(SCALING_PARENT / "scaling_runs/scripts/verify_data_bundle.py"),
        str(DATA_ROOT),
    ],
    check=True,
)
print("Data ready:", DATA_ROOT)

## 3. Optionally restore the previous Colab session

For the first session leave `UPLOAD_RESUME_BUNDLE = False`.
For every later session change it to `True`, then upload the latest
resume ZIP downloaded from the final cell of the previous session.

In [ ]:
RUN_KEY = "m30"
RUN_NAME = "m30_standard_470m_seed42"
UPLOAD_RESUME_BUNDLE = False  # Change to True from session 2 onward.

if UPLOAD_RESUME_BUNDLE:
    print(f"Upload the latest {RUN_NAME}_resume.zip")
    uploaded = files.upload()
    archives = [
        Path(name) for name in uploaded
        if name.lower().endswith(".zip") and zipfile.is_zipfile(name)
    ]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one valid resume ZIP")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "extended_runs.resume_bundle",
            "restore",
            str(archives[0].resolve()),
            "--expected-run",
            RUN_NAME,
        ],
        cwd=SCALING_PARENT,
        check=True,
    )
else:
    print("No resume bundle selected. --resume auto will start fresh "
          "only if this runtime has no existing checkpoint.")

## 4. Architecture smoke test and full-data preflight

This performs one forward/backward pass on the selected architecture,
checks for finite loss and gradients, then verifies that the full data
bundle is large enough for the complete no-repeat run.

In [ ]:
import gc, json, math, torch
from extended_runs.config import load_config
from extended_runs.model import ToolCallLanguageModel

CONFIG_PATH = EXTENDED_ROOT / "configs" / "m30_standard_470m.json"
config = load_config(CONFIG_PATH)
model = ToolCallLanguageModel(config.model).cuda().train()
sample = torch.randint(
    0, config.model.vocab_size, (1, 64), device="cuda"
)
output = model(sample[:, :-1], labels=sample[:, 1:])
output["loss"].backward()
if not torch.isfinite(output["loss"]):
    raise RuntimeError(f"Non-finite smoke-test loss: {output['loss']}")
if not all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in model.parameters()
):
    raise RuntimeError("Non-finite gradients in architecture smoke test")
print(
    f"PASS architecture smoke: loss={output['loss'].item():.4f}, "
    f"parameters={sum(p.numel() for p in model.parameters()):,}"
)
del model, sample, output
gc.collect()
torch.cuda.empty_cache()

subprocess.run(
    [
        sys.executable,
        "-m",
        "extended_runs.train",
        "--run",
        RUN_KEY,
        "--device",
        "cuda",
        "--resume",
        "auto",
        "--preflight-only",
    ],
    cwd=SCALING_PARENT,
    check=True,
)

## 5. TensorBoard

Start TensorBoard before training. It reads the same event directory
across resumed sessions because the resume ZIP preserves those files.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {EXTENDED_ROOT}/runs

## 6. Train or resume

The default session limit is 165 minutes, leaving time to package and
download the checkpoint before a four-hour Colab allocation ends.
The trainer saves a checkpoint and returns normally when the limit is
reached. You may lower this value, but do not set it above 210 minutes.

In [ ]:
SESSION_MINUTES = 165

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "extended_runs.train",
        "--run",
        RUN_KEY,
        "--device",
        "cuda",
        "--resume",
        "auto",
        "--max-session-minutes",
        str(SESSION_MINUTES),
    ],
    cwd=SCALING_PARENT,
)
if result.returncode != 0:
    raise RuntimeError(
        "Training did not exit cleanly. Inspect the traceback. If you "
        "manually interrupted the cell, run the export cell next because "
        "the trainer saves a checkpoint on KeyboardInterrupt."
    )

## 7. Inspect status, export, and download the resume bundle

Run this after every session, including the final completed session.
Keep the newest ZIP. It is all you need to resume training state in a
fresh Colab runtime, in addition to the same project and data archives.

In [ ]:
summary_path = EXTENDED_ROOT / "runs" / RUN_NAME / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    raise RuntimeError(f"Missing run summary: {summary_path}")

resume_zip = Path("/content") / f"{RUN_NAME}_resume.zip"
subprocess.run(
    [
        sys.executable,
        "-m",
        "extended_runs.resume_bundle",
        "pack",
        "--run-name",
        RUN_NAME,
        "--output",
        str(resume_zip),
    ],
    cwd=SCALING_PARENT,
    check=True,
)
print(
    f"Downloading {resume_zip.name} "
    f"({resume_zip.stat().st_size / 1024**2:.1f} MiB)"
)
files.download(str(resume_zip))